# Consolidated Leave-One-Out Evaluation

This notebook replaces the split cohort/prompt notebooks and produces the consolidated summary tables.
It uses `PRIMARY_M = 4` for the main results table, recalculates the regex baseline by default, and keeps the sensitivity table at the raw seed level.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.consolidated_loo_eval import PRIMARY_M, build_consolidated_tables, save_consolidated_tables


In [ ]:
# Main knobs
OUTPUT_DIR = Path('analysis/consolidated_loo_tables')
BOOTSTRAP_REPS = 5000
ALPHA = 0.05
RANDOM_STATE = 0
FORCE_RECALC_REGEX = True
STRICT = False
DISPLAY_FULL_TABLES = True

PRIMARY_M = 4

print(f'PRIMARY_M={PRIMARY_M}')
print(f'BOOTSTRAP_REPS={BOOTSTRAP_REPS}')
print(f'ALPHA={ALPHA}')
print(f'RANDOM_STATE={RANDOM_STATE}')
print(f'FORCE_RECALC_REGEX={FORCE_RECALC_REGEX}')
print(f'STRICT={STRICT}')
print(f'DISPLAY_FULL_TABLES={DISPLAY_FULL_TABLES}')


In [ ]:
artifacts = build_consolidated_tables(
    primary_m=PRIMARY_M,
    bootstrap_reps=BOOTSTRAP_REPS,
    alpha=ALPHA,
    random_state=RANDOM_STATE,
    force_recalc_regex=FORCE_RECALC_REGEX,
    strict=STRICT,
)
saved_paths = save_consolidated_tables(artifacts, OUTPUT_DIR)
saved_paths


In [ ]:
cohort_order = ['MIMIC', 'Indian']
prompt_order = ['short', 'long']
regime_order = ['WithUpdate', 'Random']
prompt_regime_order = ['zero-shot', 'label-only ICL', 'rationale-augmented ICL']
model_order = [
    'meta-llama/Llama-3.2-3B-Instruct',
    'mistralai/Ministral-3-3B-Instruct-2512-BF16',
    'google/gemma-3-4b-it',
    'Qwen/Qwen3-4B-Instruct-2507',
    'microsoft/MediPhi-Instruct',
    'microsoft/Phi-3.5-mini-instruct',
    'microsoft/Phi-4-mini-instruct',
    'COMMITTEE',
    'regex_baseline',
]

def sort_table(df, *, include_prompt_variant=True):
    work = df.copy()
    if 'cohort' in work.columns:
        work['cohort'] = pd.Categorical(work['cohort'], categories=cohort_order, ordered=True)
    if include_prompt_variant and 'prompt_variant' in work.columns:
        work['prompt_variant'] = pd.Categorical(work['prompt_variant'], categories=prompt_order, ordered=True)
    if 'regime' in work.columns:
        work['regime'] = pd.Categorical(work['regime'], categories=regime_order, ordered=True)
    if 'prompt_regime' in work.columns:
        work['prompt_regime'] = pd.Categorical(work['prompt_regime'], categories=prompt_regime_order, ordered=True)
    if 'model' in work.columns:
        work['model'] = pd.Categorical(work['model'], categories=model_order, ordered=True)
    sort_cols = [col for col in ['cohort', 'prompt_variant', 'model', 'regime', 'prompt_regime', 'comparison', 'class', 'm', 'seed'] if col in work.columns]
    return work.sort_values(sort_cols).reset_index(drop=True)

def display_table(df):
    if DISPLAY_FULL_TABLES:
        with pd.option_context(
            'display.max_rows', None,
            'display.max_columns', None,
            'display.max_colwidth', None,
            'display.width', None,
        ):
            display(df)
    else:
        display(df)


## Coverage And Warnings

The table below makes missing or incomplete Methods-level conditions explicit. Rows with incomplete five-seed coverage are excluded from the main collapsed tables and appear as `NaN` there.

In [ ]:
condition_status = artifacts['condition_status'].copy()
display_table(condition_status)

warnings_df = pd.DataFrame({'warning': artifacts['warnings']})
if warnings_df.empty:
    print('No warnings.')
else:
    display_table(warnings_df)


## Table 1

Main classification results. The regex baseline is prompt-agnostic, so its `prompt_variant` cell is intentionally left blank.

In [ ]:
table_1 = sort_table(artifacts['table_1'])
display_table(table_1)


## Table 2

Per-class metrics for the best overall configuration in each cohort. Two variants are shown: one where `COMMITTEE` is allowed to win, and one where it is excluded. The lookup tables are shown first because the requested Table 2 schema does not include `prompt_variant`.

In [ ]:
print('With COMMITTEE allowed')
table_2_best_configs_with_committee = sort_table(artifacts['table_2_best_configs_with_committee'])
display_table(table_2_best_configs_with_committee)

table_2_with_committee = sort_table(artifacts['table_2_with_committee'], include_prompt_variant=False)
display_table(table_2_with_committee)

print('Without COMMITTEE allowed')
table_2_best_configs_without_committee = sort_table(artifacts['table_2_best_configs_without_committee'])
display_table(table_2_best_configs_without_committee)

table_2_without_committee = sort_table(artifacts['table_2_without_committee'], include_prompt_variant=False)
display_table(table_2_without_committee)


## Table 3

Paired bootstrap comparisons on the post-majority-vote predictions.

In [ ]:
table_3 = sort_table(artifacts['table_3'])
display_table(table_3)


## Table 4

Raw per-seed sensitivity results for `m in {2, 4, 6, 8, 10}`.

In [ ]:
table_4 = sort_table(artifacts['table_4'])
display_table(table_4)


## Table 5

Model-averaged summary across the 7 SLMs only. `COMMITTEE` and `regex_baseline` are excluded.

In [ ]:
table_5 = sort_table(artifacts['table_5'])
display_table(table_5)
